# 23 — Evaluate Video Model

Loads the best checkpoint for each trained video model and runs full test-set
evaluation: classification report, confusion matrix, and inference speed.

**Predictions are utterance-level**: softmax probabilities from all clips of
each utterance are averaged before argmax.

**Depends on:** `13_train_video_model.ipynb` having been run.

**Reads from** `../experiments/video_unimodal/<version>/<model_slug>/model/`.

**Writes to each run directory:**
- `classification_report_test.json`
- `confusion_matrix_test.png`
- `eval_metrics.json`

In [ ]:
import os
import sys

_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'src')) else os.path.abspath(os.path.join(_here, '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from src.config import (
    DEVICE,
    EXP_ROOT_VIDEO,
    FACE_MELD_CLASSES,
    VERSION,
    VIDEO_CACHE_DIR,
    VIDEO_MODELS_TO_RUN,
)
from src.data.video_preprocessing import load_video_metadata, _frame_cache_dir
from src.data.video_dataset import LABEL2ID, ID2LABEL
from src.evaluation.evaluate_video import run_full_video_evaluation
from src.training.utils import slugify

print('Device          :', DEVICE)
print('Models to eval  :', VIDEO_MODELS_TO_RUN)

## Load test metadata

In [ ]:
_label2id = {cls.lower(): i for i, cls in enumerate(FACE_MELD_CLASSES)}

test_meta = load_video_metadata('test', version=VERSION)
test_df   = test_meta[
    test_meta['video_exists'] & test_meta['emotion_norm'].isin(_label2id)
].reset_index(drop=True)
test_df['label_id'] = test_df['emotion_norm'].map(_label2id).astype(int)

print(f'Test utterances : {len(test_df)}')
print(f'Test utterance count (unique): '
      f'{test_df[["dialogue_id","utterance_id"]].drop_duplicates().shape[0]}')

## Discover run directories

In [ ]:
def find_run_dir(model_name: str) -> str:
    slug = slugify(model_name)
    path = os.path.join(EXP_ROOT_VIDEO, VERSION, slug)
    assert os.path.isdir(path), (
        f'Run directory not found: {path}\n'
        'Run 13_train_video_model.ipynb first.'
    )
    return path

run_dirs = [find_run_dir(m) for m in VIDEO_MODELS_TO_RUN]
print('Evaluating:')
for d in run_dirs:
    print(' ', d)

## Run evaluation

In [ ]:
test_cache_dir = _frame_cache_dir('test', VERSION, VIDEO_CACHE_DIR)

all_results = {}

for model_name, run_dir in zip(VIDEO_MODELS_TO_RUN, run_dirs):
    print(f'\n=== {model_name} ===')
    results = run_full_video_evaluation(
        run_dir   = run_dir,
        test_df   = test_df,
        cache_dir = test_cache_dir,
        device    = DEVICE,
        version   = VERSION,
    )
    all_results[model_name] = results

## Side-by-side summary

In [ ]:
summary = pd.DataFrame([
    {
        'model':          name,
        'accuracy':       r['accuracy'],
        'macro_f1':       r['macro_f1'],
        'weighted_f1':    r['weighted_f1'],
        'infer_ms_clip':  r['inference_ms_per_clip_bs1'],
        'n_utterances':   r['num_test_utterances'],
        'total_params':   r['total_params'],
    }
    for name, r in all_results.items()
])
summary

## Confusion matrices

In [ ]:
valid_dirs = [d for d in run_dirs if os.path.exists(os.path.join(d, 'confusion_matrix_test.png'))]

if valid_dirs:
    fig, axes = plt.subplots(1, len(valid_dirs), figsize=(9 * len(valid_dirs), 7), squeeze=False)

    for ax, run_dir in zip(axes[0], valid_dirs):
        img_path = os.path.join(run_dir, 'confusion_matrix_test.png')
        ax.imshow(mpimg.imread(img_path))
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('No confusion matrix images found.')

## Per-class breakdown

In [ ]:
for model_name, run_dir in zip(VIDEO_MODELS_TO_RUN, run_dirs):
    report_path = os.path.join(run_dir, 'classification_report_test.json')
    if not os.path.exists(report_path):
        continue
    report = json.load(open(report_path))
    rows = [
        {
            'emotion':   cls,
            'precision': round(v['precision'], 4),
            'recall':    round(v['recall'],    4),
            'f1-score':  round(v['f1-score'],  4),
            'support':   int(v['support']),
        }
        for cls, v in report.items()
        if isinstance(v, dict) and 'f1-score' in v
    ]
    print(f'\n--- {model_name} ---')
    display(pd.DataFrame(rows))

## Macro-F1 comparison across modalities

Collects best dev macro-F1 from all modality experiment folders for comparison.

In [ ]:
from src.config import EXP_ROOT_FACE, EXP_ROOT_AUDIO, EXP_ROOT

def collect_eval_metrics(exp_root, version=VERSION):
    """Walk experiment root and collect eval_metrics.json files."""
    rows = []
    version_dir = os.path.join(exp_root, version)
    if not os.path.isdir(version_dir):
        return rows
    for slug in os.listdir(version_dir):
        path = os.path.join(version_dir, slug, 'eval_metrics.json')
        if os.path.exists(path):
            m = json.load(open(path))
            rows.append(m)
    return rows

all_rows = (
    collect_eval_metrics(EXP_ROOT,       VERSION) +   # text
    collect_eval_metrics(EXP_ROOT_AUDIO, VERSION) +   # audio
    collect_eval_metrics(EXP_ROOT_FACE,  VERSION) +   # face
    collect_eval_metrics(EXP_ROOT_VIDEO, VERSION)     # video
)

if all_rows:
    df_all = pd.DataFrame(all_rows)[['model_name', 'accuracy', 'macro_f1', 'weighted_f1']]
    df_all = df_all.sort_values('macro_f1', ascending=False).reset_index(drop=True)
    display(df_all)
else:
    print('No eval_metrics.json files found yet.')